# Text Summarization Project (Extractive + Abstractive)

This notebook builds two extractive summarizers and one abstractive summarizer:

- **Extractive**
  - Frequency-based sentence scoring
  - **TextRank** (via `sumy`)
- **Abstractive** (via Hugging Face Transformers)
  - Uses a pretrained seq2seq model (default: `t5-small`)

You can paste a paragraph/document and get back short summaries.

## What to run
1. Run cells **from top to bottom**.
2. If installs take time, wait until they finish.
3. The Transformers model is downloaded the first time you run that cell.


In [ ]:
# 1) Environment setup (install required libraries)
#
# Notes for Windows/Cursor:
# - Installing `torch` can take a while.
# - After installs, restart the notebook kernel if you see import/version issues.

!pip -q install nltk spacy transformers sumy gradio PyPDF2 evaluate rouge-score

# Install CPU-only torch wheel if it's not available.
try:
    import torch  # noqa: F401
except Exception:
    !pip -q install --index-url https://download.pytorch.org/whl/cpu torch

print("Installed base libraries.")


In [ ]:
# 2) Imports + NLTK data downloads

import re
from collections import Counter

import nltk

# Download NLTK resources (safe to run multiple times)
for pkg in ["punkt", "stopwords"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"NLTK download issue for {pkg}: {e}")

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

STOP_WORDS = set(stopwords.words("english"))

# Optional: spaCy (only used if you turn on lemmatization)
try:
    import spacy

    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        # Downloads the model; requires internet.
        !python -m spacy download en_core_web_sm
        nlp = spacy.load("en_core_web_sm")
except Exception:
    spacy = None
    nlp = None

print("NLTK ready. spaCy loaded:" , bool(nlp))


## 3) Data handling (sample text first)

Start with a sample paragraph, then (optionally) load text from a local `.txt` file or a `.pdf`.

For PDFs we’ll use `PyPDF2` and extract text page-by-page.


In [ ]:
# 4) Load text (sample by default)

sample_text = """
Artificial intelligence (AI) is transforming how people work and communicate. Machine learning models can identify patterns in large amounts of data, which helps systems recognize images, understand speech, and make predictions. However, using AI responsibly is critical. Models can inherit biases from the data they are trained on, and they may produce incorrect outputs when they encounter unusual inputs. Researchers and engineers therefore focus on safer training methods, better evaluation, and techniques that explain model behavior. As these tools improve, society will benefit from AI that is accurate, fair, and transparent.
""".strip()


def load_text_from_txt(path: str) -> str:
    """Load plain text from a .txt file."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_text_from_pdf(path: str) -> str:
    """Load text from a PDF using PyPDF2 (best-effort)."""
    try:
        from PyPDF2 import PdfReader
    except Exception as e:
        raise ImportError("PyPDF2 is required for PDF loading. Re-run the install cell.") from e

    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n".join(pages).strip()


# Choose your input source:
# - "sample" (no files needed)
# - "txt" (set TXT_PATH below)
# - "pdf" (set PDF_PATH below)
INPUT_MODE = "sample"

TXT_PATH = r""  # e.g. r"e:\\my_docs\\article.txt"
PDF_PATH = r""  # e.g. r"e:\\my_docs\\article.pdf"

if INPUT_MODE == "sample":
    input_text = sample_text
elif INPUT_MODE == "txt":
    if not TXT_PATH:
        raise ValueError("Set TXT_PATH before using INPUT_MODE='txt'.")
    input_text = load_text_from_txt(TXT_PATH)
elif INPUT_MODE == "pdf":
    if not PDF_PATH:
        raise ValueError("Set PDF_PATH before using INPUT_MODE='pdf'.")
    input_text = load_text_from_pdf(PDF_PATH)
else:
    raise ValueError("INPUT_MODE must be one of: 'sample', 'txt', 'pdf'.")

print("Input length (characters):", len(input_text))
print("Preview:", input_text[:300].replace("\n", " "), "...")


## 5) Preprocessing (tokenization, cleaning, stopwords)

Even before summarizing, we usually clean and tokenize text:

- **Sentence tokenization**: split the document into sentences.
- **Word tokenization**: split sentences into words/tokens.
- **Stopword removal**: remove very common words like `the`, `is`, `and`.
- **Cleaning**: normalize whitespace and keep useful word characters.


In [ ]:
# 6) Preprocess the input

def normalize_whitespace(text: str) -> str:
    text = text.replace("\r", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()


def clean_token(tok: str) -> str:
    """Keep lowercase letters/numbers and apostrophes; remove the rest."""
    tok = tok.lower()
    tok = re.sub(r"[^a-z0-9']+", "", tok)
    return tok


def tokenize_with_stopwords(text: str):
    """Return (sentences, words_without_stopwords)."""
    text = normalize_whitespace(text)
    sentences = sent_tokenize(text)

    words = []
    for s in sentences:
        for tok in word_tokenize(s):
            tok = clean_token(tok)
            if tok and tok not in STOP_WORDS and len(tok) > 1:
                words.append(tok)

    return sentences, words


sentences, words = tokenize_with_stopwords(input_text)

print("# Sentences:", len(sentences))
print("# Words after stopword removal:", len(words))
print("First 20 words:", words[:20])


## 7) Extractive summarization (sentences copied from the original text)

Extractive methods **pick** the most important sentences and return them as the summary.

### 7a) Frequency-based method
1. Count how frequently each meaningful word appears.
2. Give each sentence a score based on how many high-frequency words it contains.
3. Choose the top-scoring sentences.

### 7b) TextRank (TextRankSummarizer)
TextRank is similar in spirit to PageRank:
- Sentences are treated like nodes in a graph.
- Sentences that are similar to many others get higher rank.
- We return the top-ranked sentences.

We’ll implement both in the next cell and print the summaries.

In [ ]:
# 8) Extractive summarization implementations

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.utils import get_stop_words


def frequency_extractive_summary(text: str, n_sentences: int = 2) -> str:
    """Frequency-based extractive summarization."""
    sentences, words = tokenize_with_stopwords(text)
    if not sentences:
        return ""

    if not words:
        return " ".join(sentences[:n_sentences])

    word_freq = Counter(words)
    max_freq = max(word_freq.values())

    # Normalize word frequencies to [0, 1]
    word_score = {w: f / max_freq for w, f in word_freq.items()}

    sentence_scores = {}
    for i, s in enumerate(sentences):
        tokens = [clean_token(t) for t in word_tokenize(s)]
        tokens = [t for t in tokens if t and t not in STOP_WORDS and len(t) > 1]
        sentence_scores[i] = sum(word_score.get(t, 0.0) for t in tokens)

    # Pick top-ranked sentences, then restore original order
    top_indices = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:n_sentences]
    top_indices = sorted(top_indices)

    return " ".join(sentences[i] for i in top_indices)


def textrank_extractive_summary(text: str, n_sentences: int = 2) -> str:
    """Extractive summarization using TextRank (sumy)."""
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = TextRankSummarizer()
    summarizer.stop_words = get_stop_words("english")

    summary_sentences = summarizer(parser.document, n_sentences)
    return " ".join(str(s) for s in summary_sentences)


# Run both extractive methods
N_SENTENCES = 2

freq_summary = frequency_extractive_summary(input_text, n_sentences=N_SENTENCES)
textrank_summary = textrank_extractive_summary(input_text, n_sentences=N_SENTENCES)

print("== Frequency-based summary ==")
print(freq_summary)
print()
print("== TextRank summary ==")
print(textrank_summary)


## 9) Abstractive summarization (Hugging Face Transformers)

Abstractive summarization **rewrites** the text using a neural model, instead of only selecting existing sentences.

We’ll use a pretrained sequence-to-sequence model:
- Default here: `t5-small` (lighter and easier to run on CPU)
- Optionally you can switch to `facebook/bart-large-cnn` (often better, but heavier)

Next cell shows:
- Loading tokenizer + model
- Generating a summary with `model.generate(...)`


In [ ]:
# 10) Abstractive summary using Transformers

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


def load_seq2seq_model(model_id: str):
    """Load a seq2seq model/tokenizer to the best available device."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    model.to(device)
    model.eval()

    return tokenizer, model, device


# Pick a model:
# - "facebook/bart-large-cnn" can give strong results but is heavier.
# - "t5-small" is lighter but may produce weaker summaries unless fine-tuned for summarization.
MODEL_ID = "t5-small"

tokenizer, model, device = load_seq2seq_model(MODEL_ID)


def transformers_abstractive_summary(
    text: str,
    model_id: str = MODEL_ID,
    max_new_tokens: int = 80,
    num_beams: int = 4,
    min_new_tokens: int = 20,
):
    """Generate an abstractive summary using model.generate()."""

    # For many T5 summarization setups, a prefix helps.
    # (Harmless for BART; you can set this to "" when using non-T5 models.)
    prompt_prefix = "summarize: " if "t5" in model_id.lower() else ""
    prompt_text = prompt_prefix + text.strip().replace("\n", " ")

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )

    # Move tensors to device (works across transformers versions)
    for k, v in inputs.items():
        inputs[k] = v.to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            num_beams=num_beams,
            do_sample=False,
            length_penalty=2.0,
            early_stopping=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


abs_summary = transformers_abstractive_summary(input_text)

print("== Abstractive (Transformers) summary ==")
print(abs_summary)


## 11) Evaluation (ROUGE score)

ROUGE is a common way to evaluate summaries by comparing them to a **reference (ground-truth) summary**.

- `ROUGE-1`: overlap of unigrams (individual words)
- `ROUGE-2`: overlap of bigrams (pairs of words)
- `ROUGE-L`: overlap based on the longest common subsequence (captures sentence structure)

In this notebook we’ll use a **small, manually written reference summary** for the sample text, then compute ROUGE between:
- `frequency_extractive_summary`
- `textrank_extractive_summary`
- `transformers_abstractive_summary`


In [ ]:
# 12) Compute ROUGE scores

from rouge_score import rouge_scorer

reference_summary = (
    "Artificial intelligence uses machine learning to recognize patterns in data for tasks like images and speech. "
    "To be responsible, AI must be evaluated for bias and errors and made more accurate, fair, and transparent."
)


def compute_rouge(reference: str, hypothesis: str):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = scorer.score(reference, hypothesis)
    # Return F1 (fmeasure) for each ROUGE type
    return {k: round(v.fmeasure, 4) for k, v in scores.items()}


print("Reference summary:\n", reference_summary)

methods = {
    "Frequency (extractive)": freq_summary,
    "TextRank (extractive)": textrank_summary,
    "Transformers (abstractive)": abs_summary,
}

for name, summary in methods.items():
    print("\n==", name, "==")
    print(summary)
    print("ROUGE:", compute_rouge(reference_summary, summary))


## 13) Optional: Simple UI with Gradio

If you’d like an input/output box to test the summarizers, we can wrap the functions in **Gradio**.

When you run the next cell:
- you paste text
- choose a method (Frequency / TextRank / Transformers)
- click submit to see the summary


In [ ]:
# 14) Launch Gradio app

import gradio as gr


def summarize_with_method(text: str, method: str, n_sentences: int):
    text = (text or "").strip()
    if not text:
        return "Please enter some text."

    if method == "Frequency (extractive)":
        return frequency_extractive_summary(text, n_sentences=n_sentences)

    if method == "TextRank (extractive)":
        return textrank_extractive_summary(text, n_sentences=n_sentences)

    # Abstractive: map "n_sentences" roughly to max_new_tokens
    max_new_tokens = max(20, int(n_sentences * 25))
    return transformers_abstractive_summary(text, max_new_tokens=max_new_tokens)


METHODS = [
    "Frequency (extractive)",
    "TextRank (extractive)",
    "Transformers (abstractive)",
]

with gr.Blocks() as demo:
    gr.Markdown("# Text Summarization Demo")
    gr.Markdown("Paste text on the left. Choose a method and click **Summarize**.")

    with gr.Row():
        inp = gr.Textbox(
            label="Input text",
            lines=8,
            placeholder="Paste a paragraph or document here...",
            value="".strip(),
        )

    with gr.Row():
        method = gr.Radio(label="Summarization method", choices=METHODS, value=METHODS[0])
        n_sentences = gr.Slider(
            label="Summary length (approx.)",
            minimum=1,
            maximum=5,
            step=1,
            value=2,
        )

    out = gr.Textbox(label="Summary", lines=4)
    btn = gr.Button("Summarize")

    btn.click(fn=summarize_with_method, inputs=[inp, method, n_sentences], outputs=out)


demo.launch(debug=False, share=False)


## Troubleshooting (common issues)

- **`ModuleNotFoundError`** (nltk/sumy/transformers/etc.): re-run the install cell at the top.
- **Transformer model download takes long**: the first run downloads weights. Re-run the abstractive cell once the download finishes.
- **Out of memory (GPU/CPU)**: switch the model to a smaller one (edit `MODEL_ID`), or restart the kernel and try again.
- **spaCy model not found**: the notebook tries to download `en_core_web_sm` automatically if internet is available.
- **PDF loading fails**: ensure `PyPDF2` installed and the PDF is text-based (scanned PDFs may return empty text).

If you want, tell me your machine specs (RAM, GPU) and I can suggest the best `MODEL_ID` for your setup.